<a href="https://colab.research.google.com/github/adamcochrane/Dissertation/blob/main/gather_metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
!pip install deepeval openai bert-score

In [20]:
!git clone https://github.com/adamcochrane/Dissertation.git

fatal: destination path 'Dissertation' already exists and is not an empty directory.


In [39]:
import pandas as pd

csv_id = 3

file_name = f"/content/Dissertation/csv files/prompting-llama-advanced-{csv_id}.csv"
test_responses = pd.read_csv(file_name)

In [40]:
import csv
from google.colab import files, userdata
from openai import OpenAI
import os
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric
from bert_score import score
import ast
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

os.environ["OPENAI_API_KEY"] = userdata.get('OpenAI_key')


faithfulness_results = {}
faithfulness_reasons = {}
bertP_results = {}
bleu_results = {}
short_answers_results = {}


# FAITHFULNESS METRIC
faithful_metric = FaithfulnessMetric(
  threshold=0.7,
  model="gpt-4o-mini",
  include_reason=True
)

def getFaithfulness(prompt_id, question, response, context):
  test_case = LLMTestCase(
    input = question,
    actual_output = response,
    retrieval_context = [context]
  )
  faithful_metric.measure(test_case)
  faithfulness_results[prompt_id] = faithful_metric.score
  if faithful_metric.score < 1.0:
    faithfulness_reasons[prompt_id] = faithful_metric.reason
  else:
    faithfulness_reasons[prompt_id] = ""



#BertMetric
def getBertScore(prompt_id, response, context):
  answers = [response]
  expected_answers = [context]
  P, R, F1 = score(answers, expected_answers, lang='en', rescale_with_baseline=True)
  bertP_results[prompt_id] = P.mean().item()
  print("BERT Precision score = ",P.mean().item())



#Bleu metric
def getBleuScore(prompt_id, response, answers):
  reference = [answer.split() for answer in answers]
  candidate = response.split()
  smoothing = SmoothingFunction().method1

  print('Cumulative 1-gram: %f' % sentence_bleu(reference, candidate, weights=(1, 0, 0, 0), smoothing_function = smoothing))
  print('Cumulative 2-gram: %f' % sentence_bleu(reference, candidate, weights=(0.5, 0.5, 0, 0), smoothing_function = smoothing))
  print('Cumulative 3-gram: %f' % sentence_bleu(reference, candidate, weights=(0.33, 0.33, 0.33, 0), smoothing_function = smoothing))
  print('Cumulative 4-gram (being stored): %f' % sentence_bleu(reference, candidate, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function = smoothing))
  average_bleu_score = sentence_bleu(reference, candidate, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function = smoothing)

  bleu_results[prompt_id] = average_bleu_score



#Personal metric
def getAnswersScore(prompt_id, response, answers):
  response_lower = response.lower()
  answers_present = set()
  unique_answers = set([answer.lower() for answer in answers])

  for answer in unique_answers:
    if (answer in response_lower):
      answers_present.add(answer)

  score = len(answers_present) / len(unique_answers)
  print("Answers present score = ", score)
  short_answers_results[prompt_id] = score




for id in test_responses["prompt_id"]:
  if pd.notna(id):
    row_index = test_responses[test_responses["prompt_id"] == id].index[0]
    print("Row index = ", row_index)
    question = test_responses.iloc[row_index]["question"]
    response = test_responses.iloc[row_index]["response"]
    context = [test_responses.iloc[row_index]["retrieval_context"]]
    short_answers = ast.literal_eval(test_responses.iloc[row_index]["dataset_answers"])

    # getFaithfulness(prompt_id, question, response, context)
    getBertScore(id, response, context)
    getBleuScore(id, response, short_answers)
    getAnswersScore(id, response, short_answers)


test_responses["bert_score"] = test_responses["prompt_id"].map(bertP_results)
test_responses["bleu_score"] = test_responses["prompt_id"].map(bleu_results)
test_responses["short_answers_score"] = test_responses["prompt_id"].map(short_answers_results)
# test_responses["faithfulness_score"] = test_responses["prompt_id"].map(faithfulness_results)
# test_responses["faithfulness_reason"] = test_responses["prompt_id"].map(faithfulness_reasons)

file_name = f"/content/Dissertation/csv files/prompting-llama-advanced-{csv_id}-metrics.csv"
test_responses.to_csv(file_name, index=False)
files.download(file_name)







Row index =  0


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERT Precision score =  0.71693354845047
Cumulative 1-gram: 0.076923
Cumulative 2-gram: 0.025318
Cumulative 3-gram: 0.018733
Cumulative 4-gram (being stored): 0.015537
Answers present score =  0.5
Row index =  1


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERT Precision score =  0.16640742123126984
Cumulative 1-gram: 0.019802
Cumulative 2-gram: 0.004450
Cumulative 3-gram: 0.002880
Cumulative 4-gram (being stored): 0.002126
Answers present score =  1.0
Row index =  2


...

BERT Precision score =  0.3290056884288788
Cumulative 1-gram: 0.050847
Cumulative 2-gram: 0.029609
Cumulative 3-gram: 0.012070
Cumulative 4-gram (being stored): 0.007239
Answers present score =  0.6666666666666666
Row index =  2386


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERT Precision score =  0.3394748866558075
Cumulative 1-gram: 0.101695
Cumulative 2-gram: 0.072526
Cumulative 3-gram: 0.021802
Cumulative 4-gram (being stored): 0.011330
Answers present score =  1.0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>